# MLP Inference — Single WAV Snippet Classifier (Fine-tuned MERT)

Load a saved MLP head and a **fine-tuned MERT backbone**, pass in a WAV file with a
time window, and see the top predicted pieces.

**Pipeline:**
1. Load WAV → slice `[start_sec, end_sec]` → resample to 24 kHz mono  
2. Embed with **your fine-tuned MERT** (mean-pool over time frames, same as `embed.py`)  
3. Forward through saved MLP  
4. Display top-K predictions with confidence scores

**Fine-tuned MERT loading:**  
Set `FINETUNED_MERT_PATH` to one of:
- A directory saved with `model.save_pretrained(path)` — loaded via `AutoModel.from_pretrained`
- A `.pt` file containing a raw state dict — loaded into a fresh `AutoModel` base and then overwritten
- `None` — falls back to the pretrained HuggingFace weights (same behaviour as the original notebook)

In [1]:
# Mount drive (Colab only — skip if running locally)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%cd drive/MyDrive/stat-4830

/content/drive/MyDrive/stat-4830


In [ ]:
%ls

## 1. Configuration

Set paths and input here before running.

In [3]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────

# Path to the WAV file you want to classify.
WAV_PATH = "demo/bwv_248_64_yt.wav"

# Time window within the file to use as the snippet (seconds).
START_SEC = 0.0
END_SEC   = 5.0   # set to None to use the whole file from START_SEC onwards

# Speed multiplier — stretches time WITHOUT changing pitch.
#   1.0  = no change
#   1.5  = 50% faster (same pitch)
#   2.0  = double speed (same pitch)
#   0.75 = 25% slower (same pitch)
# Set to 1.0 to disable entirely (no processing overhead).
SPEED_FACTOR = 0.8

# Where the saved MLP + label encoder live.
MODEL_DIR = "perturb/embeddings_new"   # e.g. perturb/embeddings_new

# ── Fine-tuned MERT ───────────────────────────────────────────────────────────
#
# Option A — HuggingFace save_pretrained directory:
#   FINETUNED_MERT_PATH = "my_finetuned_mert"
#
# Option B — raw state-dict .pt file:
#   FINETUNED_MERT_PATH = "checkpoints/mert_finetuned.pt"
#
# Option C — fall back to pretrained HuggingFace weights (original behaviour):
#   FINETUNED_MERT_PATH = None
#
FINETUNED_MERT_PATH = "perturb/finetune/finetune_best_20260419_225041_4layers.pt"   # <-- set this to your fine-tuned model path

# How many top guesses to display.
TOP_K = 10

# Device for MERT inference.
DEVICE = "cuda"   # "cpu" if no GPU available

# Base MERT model string — used for the processor and as the architecture
# base when loading a state-dict .pt file.  Must match what was used
# during embed.py / fine-tuning.
EMBEDDING_MODEL = "m-a-p/MERT-v1-95M"
SAMPLE_RATE     = 24_000

## 2. Imports

In [4]:
import os
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 3. MLP Definition

Same class as in the training notebook — required to reconstruct the model from
its saved weights.

In [5]:
class MLP(nn.Module):
    """
    Configurable MLP for multi-class classification.
    Must match the definition in w12_mlp_search.ipynb exactly.
    """
    def __init__(
        self,
        input_dim:   int,
        hidden_dims: list,
        n_classes:   int,
        dropout_p:   float,
        batch_norm:  bool = True,
    ):
        super().__init__()
        layers = []
        in_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))
            in_dim = h

        layers.append(nn.Linear(in_dim, n_classes))
        self.net = nn.Sequential(*layers)

        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 4. Load Saved MLP + Label Encoder

In [6]:
model_dir = Path(MODEL_DIR)

# ── Config / metadata ─────────────────────────────────────────────────────────
ckpt = torch.load(model_dir / "mlp_best_config.pt", weights_only=True)
print("Loaded MLP config:")
print(f"  Architecture : {ckpt['input_dim']} → {'→'.join(str(h) for h in ckpt['hidden_dims'])} → {ckpt['n_classes']}")
print(f"  Dropout      : {ckpt['dropout_p']}")
print(f"  Batch norm   : {ckpt['batch_norm']}")
print(f"  Parameters   : {ckpt['n_params']:,}")
print(f"  Saved top-1  : {ckpt['best_top1']:.4f}  ({ckpt['best_top1']*100:.1f}%)")

# ── Reconstruct and load weights ──────────────────────────────────────────────
mlp = MLP(
    input_dim   = ckpt["input_dim"],
    hidden_dims = ckpt["hidden_dims"],
    n_classes   = ckpt["n_classes"],
    dropout_p   = ckpt["dropout_p"],
    batch_norm  = ckpt["batch_norm"],
)
mlp.load_state_dict(torch.load(model_dir / "mlp_best.pt", weights_only=True))
mlp.eval()
print("\nMLP weights loaded.")

# ── Label encoder ─────────────────────────────────────────────────────────────
with open(model_dir / "label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print(f"Label encoder loaded: {len(le.classes_)} classes")

Loaded MLP config:
  Architecture : 768 → 2048→512 → 405
  Dropout      : 0.5
  Batch norm   : True
  Parameters   : 2,836,885
  Saved top-1  : 0.4466  (44.7%)

MLP weights loaded.
Label encoder loaded: 405 classes


## 5. Load MERT (Fine-tuned or Pretrained)

- If `FINETUNED_MERT_PATH` points to a **`save_pretrained` directory**, loads directly with `AutoModel.from_pretrained`.
- If it points to a **`.pt` state-dict file**, initialises the base architecture from `EMBEDDING_MODEL` and overwrites with your weights.
- If `None`, falls back to the original pretrained HuggingFace weights.

The processor is always loaded from `EMBEDDING_MODEL` (it's architecture-only, not weights).

In [7]:
from transformers import AutoModel, AutoProcessor

# ── Processor (always from base model — architecture only, no fine-tuned weights) ──
print(f"Loading processor from {EMBEDDING_MODEL} ...")
processor = AutoProcessor.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
print("Processor loaded.")

# ── MERT backbone ─────────────────────────────────────────────────────────────
if FINETUNED_MERT_PATH is None:
    # ── Fallback: original pretrained weights ──────────────────────────────────
    print(f"\nFINETUNED_MERT_PATH is None — loading pretrained {EMBEDDING_MODEL} ...")
    mert = AutoModel.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
    print("Pretrained MERT loaded.")

elif os.path.isdir(FINETUNED_MERT_PATH):
    # ── Option A: save_pretrained directory ────────────────────────────────────
    print(f"\nLoading fine-tuned MERT from directory: {FINETUNED_MERT_PATH} ...")
    mert = AutoModel.from_pretrained(
        FINETUNED_MERT_PATH,
        trust_remote_code=True,
    )
    print("Fine-tuned MERT loaded from save_pretrained directory.")

elif os.path.isfile(FINETUNED_MERT_PATH) and FINETUNED_MERT_PATH.endswith(".pt"):
    # ── Option B: raw state-dict .pt file ──────────────────────────────────────
    # Initialise the architecture from the base model, then overwrite weights.
    print(f"\nInitialising {EMBEDDING_MODEL} architecture for state-dict loading ...")
    mert = AutoModel.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
    print(f"Loading fine-tuned state dict from: {FINETUNED_MERT_PATH} ...")
    state_dict = torch.load(FINETUNED_MERT_PATH, map_location="cpu", weights_only=True)
    # Handle checkpoints that wrap the model state dict under a key
    # (e.g. Lightning saves {"state_dict": ..., "epoch": ...})
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
        # Strip any "model." prefix added by wrappers
        state_dict = {k.removeprefix("model."): v for k, v in state_dict.items()}
    missing, unexpected = mert.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  ⚠ Missing keys  ({len(missing)}): {missing[:5]}{'...' if len(missing)>5 else ''}")
    if unexpected:
        print(f"  ⚠ Unexpected keys ({len(unexpected)}): {unexpected[:5]}{'...' if len(unexpected)>5 else ''}")
    print("Fine-tuned MERT state dict loaded.")

else:
    raise FileNotFoundError(
        f"FINETUNED_MERT_PATH={FINETUNED_MERT_PATH!r} is neither a directory nor a .pt file."
    )

mert = mert.to(DEVICE)
mert.eval()
print(f"\nMERT on {DEVICE}, eval mode — ready.")

Loading processor from m-a-p/MERT-v1-95M ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

...

modeling_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Loading fine-tuned state dict from: perturb/finetune/finetune_best_20260419_225041_4layers.pt ...
  ⚠ Missing keys  (211): ['masked_spec_embed', 'feature_extractor.conv_layers.0.conv.weight', 'feature_extractor.conv_layers.0.layer_norm.weight', 'feature_extractor.conv_layers.0.layer_norm.bias', 'feature_extractor.conv_layers.1.conv.weight']...
  ⚠ Unexpected keys (7): ['mert_state', 'head_state', 'unfreeze_layers', 'top1', 'top5']...
Fine-tuned MERT state dict loaded.

MERT on cuda, eval mode — ready.


## 6. Audio Helpers

Load the WAV, slice to the requested window, and embed with the MERT backbone
(mean-pool over the last hidden state time dimension — identical to `embed.py`).

In [8]:
def load_snippet(
    wav_path: str,
    start_sec: float,
    end_sec,
    target_sr: int = SAMPLE_RATE,
    speed_factor: float = 1.0,
) -> np.ndarray:
    """
    Load a WAV file, extract the [start_sec, end_sec] window, resample to
    target_sr, and optionally time-stretch by speed_factor.

    Time stretching uses librosa's phase-vocoder (time_stretch), which
    speeds up or slows down audio WITHOUT changing pitch. A factor of 1.5
    makes the snippet 1.5x faster; 0.75 makes it slower.

    Note: the time window you specify is always in terms of the ORIGINAL
    file's timestamps — stretching is applied after slicing.
    """
    try:
        import soundfile as sf
        waveform, sr = sf.read(wav_path, dtype="float32", always_2d=False)
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)  # stereo → mono
    except Exception:
        import librosa
        waveform, sr = librosa.load(wav_path, sr=None, mono=True)

    # Slice to [start_sec, end_sec] in the original sample rate.
    start_sample = int(start_sec * sr)
    end_sample   = int(end_sec * sr) if end_sec is not None else len(waveform)
    end_sample   = min(end_sample, len(waveform))
    waveform     = waveform[start_sample:end_sample]

    if len(waveform) == 0:
        raise ValueError(f"Empty slice: start={start_sec}s, end={end_sec}s is out of range.")

    # Resample to MERT's expected sample rate if necessary.
    if sr != target_sr:
        import librosa
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # Pitch-preserving time stretch.
    if speed_factor != 1.0:
        import librosa
        waveform = librosa.effects.time_stretch(waveform, rate=speed_factor)
        print(f"Time-stretched ×{speed_factor} (pitch preserved)")

    duration = len(waveform) / target_sr
    print(f"Loaded snippet: {duration:.2f}s  ({len(waveform):,} samples @ {target_sr} Hz)")
    return waveform


def embed_snippet(
    waveform: np.ndarray,
    model,
    proc,
    device: str = DEVICE,
    sr: int = SAMPLE_RATE,
) -> np.ndarray:
    """
    Pass a single waveform through the MERT backbone (fine-tuned or pretrained)
    and return the mean-pooled last-hidden-state embedding vector.

    This is identical to embed.py: mean-pool over the time dimension of
    last_hidden_state → (hidden_dim,) numpy array.
    """
    inputs = proc(
        [waveform],           # processor expects a list
        sampling_rate=sr,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # last_hidden_state: (1, time_frames, hidden_dim) → mean over time → (hidden_dim,)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
    print(f"Embedding shape: {embedding.shape}")
    return embedding

## 7. Classify

In [9]:
# ── Step 1: load and slice the audio ─────────────────────────────────────────
waveform = load_snippet(WAV_PATH, START_SEC, END_SEC, speed_factor=SPEED_FACTOR)

# ── Step 2: embed with fine-tuned (or pretrained) MERT ───────────────────────
embedding = embed_snippet(waveform, mert, processor)

# ── Step 3: MLP forward pass ──────────────────────────────────────────────────
x = torch.from_numpy(embedding).float().unsqueeze(0)  # (1, D)
with torch.no_grad():
    logits = mlp(x)                        # (1, n_classes)
    probs  = F.softmax(logits, dim=1)[0]   # (n_classes,)

# ── Step 4: top-K results ─────────────────────────────────────────────────────
topk_probs, topk_indices = probs.topk(TOP_K)
topk_probs   = topk_probs.numpy()
topk_indices = topk_indices.numpy()
topk_classes = le.inverse_transform(topk_indices)

mert_source = (
    f"fine-tuned ({FINETUNED_MERT_PATH})"
    if FINETUNED_MERT_PATH is not None
    else f"pretrained ({EMBEDDING_MODEL})"
)

print()
print("=" * 60)
print(f"Input : {Path(WAV_PATH).name}  [{START_SEC}s – {END_SEC}s]")
print(f"MERT  : {mert_source}")
print("=" * 60)
print(f"  {'Rank':<5}  {'Confidence':>10}   Class")
print(f"  {'-'*4}  {'-'*10}   {'-'*40}")
for rank, (cls, prob) in enumerate(zip(topk_classes, topk_probs), start=1):
    marker = "  ◀ top guess" if rank == 1 else ""
    print(f"  {rank:<5}  {prob*100:>9.2f}%   {cls}{marker}")
print("=" * 60)

Time-stretched ×0.8 (pitch preserved)
Loaded snippet: 6.25s  (150,000 samples @ 24000 Hz)
Embedding shape: (768,)

Input : bwv_248_64_yt.wav  [0.0s – 5.0s]
MERT  : fine-tuned (perturb/finetune/finetune_best_20260419_225041_4layers.pt)
  Rank   Confidence   Class
  ----  ----------   ----------------------------------------
  1          90.51%   bach__bwv248.64-6  ◀ top guess
  2           9.44%   bach__bwv79.3
  3           0.03%   bach__bwv175.7
  4           0.02%   bach__bwv248.9-1
  5           0.00%   bach__bwv261
  6           0.00%   bach__riemenschneider016
  7           0.00%   bach__bwv248.42-4
  8           0.00%   bach__bwv378
  9           0.00%   bach__bwv226.2
  10          0.00%   bach__bwv136.6


## 8. Try a Different Window (Optional)

Adjust `START_SEC` / `END_SEC` at the top (cell 1) and re-run cells 7 + 8, or
use the quick override below to test a second window without changing the config.

In [15]:
import librosa
import soundfile as sf

y, sr = librosa.load("demo/bwv_248_64_yt.mp3", sr=24000, mono=True)
sf.write("demo/bwv_248_64_yt.wav", y, sr)

In [17]:
_WAV   = "demo/bwv_248_64_yt.wav"
_START = 0.0
_END   = 5.0
SPEED_FACTOR = 1.0

_waveform  = load_snippet(_WAV, _START, _END, speed_factor=SPEED_FACTOR)
_embedding = embed_snippet(_waveform, mert, processor)

_x = torch.from_numpy(_embedding).float().unsqueeze(0)
with torch.no_grad():
    _probs = F.softmax(mlp(_x), dim=1)[0]

_topk_probs, _topk_idx = _probs.topk(TOP_K)
_topk_classes = le.inverse_transform(_topk_idx.numpy())

print()
print("=" * 60)
print(f"Input : {Path(_WAV).name}  [{_START}s – {_END}s]")
print("=" * 60)
for rank, (cls, prob) in enumerate(zip(_topk_classes, _topk_probs.numpy()), start=1):
    marker = "  ◀ top guess" if rank == 1 else ""
    print(f"  {rank:<5}  {prob*100:>9.2f}%   {cls}{marker}")
print("=" * 60)

Loaded snippet: 5.00s  (120,000 samples @ 24000 Hz)
Embedding shape: (768,)

Input : bwv_248_64_yt.wav  [0.0s – 5.0s]
  1          88.70%   bach__bwv79.3  ◀ top guess
  2          10.11%   bach__bwv248.64-6
  3           1.18%   bach__bwv175.7
  4           0.01%   bach__bwv261
  5           0.00%   bach__bwv248.9-1
  6           0.00%   bach__bwv248.42-4
  7           0.00%   bach__bwv136.6
  8           0.00%   bach__bwv226.2
  9           0.00%   bach__bwv172.6
  10          0.00%   bach__bwv69.6


In [ ]:
%ls demo